In [71]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import numpy as np 
import PIL.Image as image
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
import os


**We will treat different sides of the iris as different subjects , This is a check for the balance of the data classes , and we can see that the data is reasonably balanced**

In [ ]:
data_path = 'DataSet/CASIA-Iris-Thousand' 
folders = os.listdir(data_path)
total_classes = 0
image_counts = {}
for fold in folders:
    sub_path = os.path.join(data_path, fold)
    if os.path.isdir(sub_path):
        for side in ['L', 'R']:
            side_path = os.path.join(sub_path, side)
            if os.path.exists(side_path):
                total_classes += 1
                imgs = os.listdir(side_path)
                image_counts[side_path] = len(imgs)
print(f"Total Unique Irises (Classes): {total_classes}")
print(f"Average Images per Class: {sum(image_counts.values())/len(image_counts):.2f}")
print(f"Min images: {min(image_counts.values())}, Max images: {max(image_counts.values())}")

Total Unique Irises (Classes): 2000
Average Images per Class: 10.01
Min images: 10, Max images: 11


In [64]:
label_mapping=pd.read_csv(r"DataSet\iris_thousands.csv",index_col=0)

In [65]:
label_mapping

,Label,ImagePath
0,437-R,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
1,437-R,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
2,437-R,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
3,437-R,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
4,437-R,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
...,...,...
19995,715-L,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
19996,715-L,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
19997,715-L,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
19998,715-L,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...


In [66]:
label_mapping['ImagePath'] = label_mapping['ImagePath'].str.replace('/kaggle/input/casia-iris-thousand/','DataSet/', 
    regex=False)

In [67]:
label_mapping

,Label,ImagePath
0,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R06.jpg
1,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R09.jpg
2,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R07.jpg
3,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R02.jpg
4,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R03.jpg
...,...,...
19995,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L04.jpg
19996,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L09.jpg
19997,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L08.jpg
19998,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L06.jpg


**Now we need to label encode the classes to be able to use them in the model training**

In [68]:
le = LabelEncoder()
label_mapping

,Label,ImagePath
0,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R06.jpg
1,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R09.jpg
2,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R07.jpg
3,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R02.jpg
4,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R03.jpg
...,...,...
19995,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L04.jpg
19996,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L09.jpg
19997,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L08.jpg
19998,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L06.jpg


In [69]:
label_mapping.sort_values(by='Label',inplace=True)
label_mapping['encoded_label'] = le.fit_transform(label_mapping['Label'])
label_mapping.to_csv(r"DataSet\mapped.csv")

In [70]:
label_mapping.head(20)

,Label,ImagePath,encoded_label
3331,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L00.jpg,0
3338,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L08.jpg,0
3337,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L07.jpg,0
3336,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L02.jpg,0
3335,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L06.jpg,0
3334,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L05.jpg,0
3333,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L01.jpg,0
3332,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L09.jpg,0
3339,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L04.jpg,0
3330,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L03.jpg,0
